# 🎬 Video Object Replacement — Google Colab

Этот ноутбук заменяет объекты на видео с помощью:
- **Grounding DINO** — находит объект по текстовому описанию
- **SAM2** — сегментирует и отслеживает объект по всем кадрам
- **SDXL + ControlNet** — генерирует новый объект с учётом формы и освещения

### Инструкция:
1. **Runtime → Change runtime type → T4 GPU** (обязательно!)
2. Запускай ячейки по порядку (`Shift+Enter`)
3. Загрузи своё видео в ячейке «Загрузка видео»
4. Укажи что найти и что сгенерировать в ячейке «Параметры»
5. Запусти пайплайн и скачай результат

> ⚠️ Установка зависимостей занимает ~5-10 минут (один раз за сессию)

## 0. Проверка GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU не найден!\n"
        "Перейди: Runtime → Change runtime type → Hardware accelerator → GPU (T4)"
    )

print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"   CUDA: {torch.version.cuda}")

## 1. Установка зависимостей

> Запускается один раз. Займёт 5-10 минут.

In [ ]:
import subprocess, sys

def run(cmd, **kw):
    print(f"▶ {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.returncode != 0:
        print(result.stderr[-2000:] if result.stderr else "(no stderr)")
    else:
        # show last few lines of stdout for progress feedback
        lines = (result.stdout or "").strip().splitlines()
        if lines:
            print("\n".join(lines[-3:]))
    return result.returncode

# Core ML / vision packages
run("pip install -q diffusers==0.30.3 transformers==4.44.2 accelerate xformers --extra-index-url https://download.pytorch.org/whl/cu121")
run("pip install -q imageio[ffmpeg] opencv-python-headless supervision")

# Grounding DINO
run("pip install -q groundingdino-py")

# SAM2
run("pip install -q 'git+https://github.com/facebookresearch/sam2.git'")

print("\n✅ Все зависимости установлены!")

## 2. Загрузка весов моделей

SAM2 (~900 MB) и Grounding DINO (~700 MB) скачиваются один раз.

In [ ]:
import os

MODELS_DIR = "/content/models"
os.makedirs(MODELS_DIR, exist_ok=True)

SAM2_CHECKPOINT  = f"{MODELS_DIR}/sam2_hiera_large.pt"
GDINO_CHECKPOINT = f"{MODELS_DIR}/groundingdino_swint_ogc.pth"

def download(url, dest):
    if os.path.exists(dest):
        size_mb = os.path.getsize(dest) / 1e6
        print(f"  ✓ уже есть: {dest} ({size_mb:.0f} MB)")
        return
    print(f"  ⬇ Качаю {os.path.basename(dest)}...")
    os.system(f'wget -q --show-progress -O "{dest}" "{url}"')
    size_mb = os.path.getsize(dest) / 1e6
    print(f"  ✓ готово: {size_mb:.0f} MB")

print("SAM2 Large:")
download(
    "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt",
    SAM2_CHECKPOINT,
)

print("Grounding DINO:")
download(
    "https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth",
    GDINO_CHECKPOINT,
)

# SAM2 config: ВСЕГДА передаём относительный путь — Hydra ищет
# конфиги относительно пакета sam2 (pkg://sam2), абсолютный путь сломает загрузку.
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"

# Проверяем, что конфиг действительно есть в установленном пакете
import sam2 as _sam2_pkg
_pkg_config = os.path.join(os.path.dirname(_sam2_pkg.__file__), "configs", "sam2.1", "sam2.1_hiera_l.yaml")
if os.path.exists(_pkg_config):
    print(f"  ✓ SAM2 config найден в пакете: {_pkg_config}")
else:
    print(f"  ⚠ Конфиг не найден по пути: {_pkg_config}")
    print("    Попробуй переустановить SAM2: pip install 'git+https://github.com/facebookresearch/sam2.git'")
print(f"  Будет использован config: '{SAM2_CONFIG}' (относительный путь для Hydra)")

print("\n✅ Модели готовы!")

## 3. Код пайплайна

In [ ]:
import cv2
import numpy as np
import torch
import tempfile
import os
import imageio.v2 as imageio
from PIL import Image

DEVICE = "cuda"
DTYPE  = torch.float16
WIDTH  = 1024
HEIGHT = 1024

print(f"Device: {DEVICE} | dtype: {DTYPE}")

In [ ]:
# ── Извлечение кадров ──────────────────────────────────────────────────

def extract_frames(video_path: str, max_frames: int = None, start_frame: int = 0):
    """Читает кадры из видео, ресайзит до WIDTH×HEIGHT.

    start_frame — с какого кадра начинать (для центрирования вокруг объекта).
    max_frames  — сколько кадров взять (None = все до конца).
    """
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if start_frame > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (WIDTH, HEIGHT))
        frames.append(frame)
        if max_frames and len(frames) >= max_frames:
            break

    cap.release()
    print(f"  Извлечено {len(frames)} кадров (с {start_frame} по {start_frame+len(frames)-1}) из {total} @ {fps:.2f} FPS")
    return frames, fps


def scan_video_for_object(video_path: str, gdino_model, text_prompt: str,
                           box_threshold: float, text_threshold: float,
                           scan_frames: int = 30):
    """Сканирует ВСЁ видео целиком (без загрузки в RAM) и возвращает лучшую детекцию.

    Читает только нужные кадры через cv2, не загружая весь ролик.
    Возвращает (best_confidence, best_frame_idx, bbox, phrase) или None.
    """
    cap = cv2.VideoCapture(video_path)
    fps   = cap.get(cv2.CAP_PROP_FPS) or 24.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    # Равномерная выборка по всей длине видео
    if scan_frames >= total:
        probe_indices = list(range(total))
    else:
        step = total / scan_frames
        probe_indices = [int(i * step) for i in range(scan_frames)]

    print(f"  Видео: {total} кадров @ {fps:.1f} FPS  |  проверяю {len(probe_indices)} кадров")

    all_detections = []

    cap = cv2.VideoCapture(video_path)
    prev_idx = -1
    for fi in probe_indices:
        # Прыгаем к нужному кадру только если нужно (seek дорогой для больших скачков)
        if fi != prev_idx + 1:
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame = cap.read()
        if not ret:
            continue
        prev_idx = fi

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_rgb = cv2.resize(frame_rgb, (WIDTH, HEIGHT))

        result = detect_bbox(gdino_model, frame_rgb, text_prompt, box_threshold, text_threshold)
        if result is not None:
            if len(result) == 3:
                bbox, phrase, conf = result
            else:
                bbox, phrase = result
                conf = 0.5
            all_detections.append((conf, fi, bbox, phrase, frame_rgb))
            print(f"  Кадр {fi:4d}/{total}: '{phrase}'  conf={conf:.3f}  ✓")
        else:
            print(f"  Кадр {fi:4d}/{total}: не найдено")

    cap.release()
    return all_detections


# ── Grounding DINO ─────────────────────────────────────────────────────

def load_grounding_dino(checkpoint_path: str):
    try:
        import groundingdino
        from groundingdino.util.inference import load_model
        config_path = os.path.join(
            os.path.dirname(groundingdino.__file__),
            "config", "GroundingDINO_SwinT_OGC.py",
        )
        if not os.path.exists(config_path):
            raise FileNotFoundError(f"Config не найден: {config_path}")
        model = load_model(config_path, checkpoint_path, device=DEVICE)
        print(f"  ✓ Grounding DINO загружен")
        return model
    except Exception as e:
        raise RuntimeError(f"Ошибка загрузки Grounding DINO: {e}")


def detect_bbox(model, image_np: np.ndarray, text_prompt: str,
                box_threshold: float = 0.35, text_threshold: float = 0.25):
    """Возвращает (x1,y1,x2,y2), phrase, confidence или None если не найдено."""
    import torchvision.transforms as T
    from groundingdino.util.inference import predict

    transform = T.Compose([
        T.Resize((800, 800)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    image_tensor = transform(Image.fromarray(image_np))
    h, w = image_np.shape[:2]

    boxes, logits, phrases = predict(
        model=model, image=image_tensor, caption=text_prompt,
        box_threshold=box_threshold, text_threshold=text_threshold,
        device=DEVICE,
    )

    if len(boxes) == 0:
        return None

    best = int(logits.argmax())
    cx, cy, bw, bh = boxes[best].tolist()
    x1 = max(0, int((cx - bw / 2) * w))
    y1 = max(0, int((cy - bh / 2) * h))
    x2 = min(w, int((cx + bw / 2) * w))
    y2 = min(h, int((cy + bh / 2) * h))
    confidence = float(logits[best])
    return (x1, y1, x2, y2), phrases[best], confidence


# ── SAM2 Image Predictor (для маски на первом кадре) ───────────────────

def build_sam2_image_predictor(checkpoint: str, config: str):
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    model = build_sam2(config, checkpoint, device=DEVICE)
    predictor = SAM2ImagePredictor(model)
    print("  ✓ SAM2 Image Predictor загружен")
    return predictor


def get_mask_from_bbox(predictor, image_np: np.ndarray, bbox):
    x1, y1, x2, y2 = bbox
    predictor.set_image(image_np)
    masks, _, _ = predictor.predict(
        box=np.array([x1, y1, x2, y2]),
        multimask_output=False,
    )
    return masks[0].astype(np.uint8) * 255


# ── SAM2 Video Predictor (отслеживание по всем кадрам) ─────────────────

def build_sam2_video_predictor(checkpoint: str, config: str):
    from sam2.build_sam import build_sam2_video_predictor
    predictor = build_sam2_video_predictor(config, checkpoint, device=DEVICE)
    print("  ✓ SAM2 Video Predictor загружен")
    return predictor


def _write_frames_to_dir(frames, directory):
    for i, frame in enumerate(frames):
        bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        cv2.imwrite(os.path.join(directory, f"{i:05d}.jpg"), bgr,
                    [cv2.IMWRITE_JPEG_QUALITY, 95])


def track_masks_video(predictor, frames, initial_mask, initial_frame_idx=0):
    n = len(frames)
    with tempfile.TemporaryDirectory() as tmpdir:
        _write_frames_to_dir(frames, tmpdir)
        with torch.inference_mode():
            state = predictor.init_state(tmpdir)
            predictor.reset_state(state)
            predictor.add_new_mask(
                inference_state=state,
                frame_idx=initial_frame_idx,
                obj_id=1,
                mask=initial_mask > 128,
            )
            tracked = [None] * n
            for out_idx, obj_ids, logits in predictor.propagate_in_video(state):
                if 1 not in obj_ids:
                    continue
                ch = list(obj_ids).index(1)
                binary = (logits[ch] > 0.0).squeeze(0).cpu().numpy()
                tracked[out_idx] = binary.astype(np.uint8) * 255

    empty = np.zeros(initial_mask.shape, dtype=np.uint8)
    return [m if m is not None else empty.copy() for m in tracked]


def fill_mask_gaps(masks, max_gap=2):
    n = len(masks)
    filled = [m.copy() for m in masks]
    i = 0
    while i < n:
        if filled[i].any():
            i += 1
            continue
        j = i
        while j < n and not filled[j].any():
            j += 1
        gap_len = j - i
        has_left  = i > 0 and filled[i - 1].any()
        has_right = j < n and filled[j].any()
        if gap_len <= max_gap and has_left and has_right:
            for k in range(i, j):
                t = (k - i + 1) / (gap_len + 1)
                blended = (1 - t) * filled[i-1].astype(np.float32) + t * filled[j].astype(np.float32)
                filled[k] = (blended > 127).astype(np.uint8) * 255
        i = j
    return filled


# ── Diffusion Pipeline ─────────────────────────────────────────────────

def load_pipeline():
    from diffusers import ControlNetModel, StableDiffusionXLControlNetInpaintPipeline

    print("  Загружаю ControlNet...")
    controlnet = ControlNetModel.from_pretrained(
        "diffusers/controlnet-canny-sdxl-1.0",
        torch_dtype=DTYPE,
    ).to(DEVICE)

    print("  Загружаю SDXL Inpainting...")
    pipe = StableDiffusionXLControlNetInpaintPipeline.from_pretrained(
        "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
        controlnet=controlnet,
        torch_dtype=DTYPE,
    ).to(DEVICE)

    try:
        pipe.enable_xformers_memory_efficient_attention()
        print("  ✓ xformers включён")
    except Exception:
        pass

    pipe.set_progress_bar_config(disable=True)
    print("  ✓ Diffusion pipeline загружен")
    return pipe


def make_canny(image_np):
    gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    return Image.fromarray(np.stack([edges] * 3, axis=-1))


def inpaint_frame(pipe, image_np, mask_np, control_image, prompt, negative_prompt, seed=42):
    image_pil = Image.fromarray(image_np)
    mask_pil  = Image.fromarray(mask_np)
    generator = torch.Generator(DEVICE).manual_seed(seed)
    with torch.no_grad():
        result = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            image=image_pil,
            mask_image=mask_pil,
            control_image=control_image,
            guidance_scale=7.5,
            num_inference_steps=30,
            generator=generator,
            output_type="pil",
        )
    return np.array(result.images[0])


# ── Пост-обработка ─────────────────────────────────────────────────────

def compute_flow(frame1, frame2):
    g1 = cv2.cvtColor(frame1, cv2.COLOR_RGB2GRAY)
    g2 = cv2.cvtColor(frame2, cv2.COLOR_RGB2GRAY)
    no_flow = np.zeros((*g1.shape, 2), dtype=np.float32)
    return cv2.calcOpticalFlowFarneback(g1, g2, no_flow, 0.5, 3, 15, 3, 5, 1.2, 0)


def _remap(src, flow, interp):
    h, w = flow.shape[:2]
    gx, gy = np.meshgrid(np.arange(w), np.arange(h))
    map_x = (gx + flow[..., 0]).astype(np.float32)
    map_y = (gy + flow[..., 1]).astype(np.float32)
    return cv2.remap(src, map_x, map_y, interp)


def apply_temporal_consistency(current, prev_result, mask, flow, alpha=0.85):
    if prev_result is None or flow is None:
        return current
    warped_prev = _remap(prev_result, flow, cv2.INTER_LINEAR)
    mask_f = (mask[..., np.newaxis] / 255.0).astype(np.float32)
    blended = alpha * current.astype(np.float32) + (1.0 - alpha) * warped_prev.astype(np.float32)
    result = current.astype(np.float32) * (1.0 - mask_f) + blended * mask_f
    return result.clip(0, 255).astype(np.uint8)


def color_harmonize(original, inpainted, mask):
    mask_bool = mask > 128
    surround = ~mask_bool
    if mask_bool.sum() < 10 or surround.sum() < 10:
        return inpainted
    orig_lab = cv2.cvtColor(original, cv2.COLOR_RGB2LAB).astype(np.float32)
    inp_lab  = cv2.cvtColor(inpainted, cv2.COLOR_RGB2LAB).astype(np.float32)
    result_lab = inp_lab.copy()
    for c in range(3):
        surr_vals = orig_lab[..., c][surround]
        inp_vals  = inp_lab[..., c][mask_bool]
        surr_mean, surr_std = surr_vals.mean(), surr_vals.std() + 1e-6
        inp_mean,  inp_std  = inp_vals.mean(),  inp_vals.std()  + 1e-6
        adjusted = (inp_vals - inp_mean) / inp_std * surr_std + surr_mean
        result_lab[..., c][mask_bool] = adjusted.clip(0, 255)
    return cv2.cvtColor(result_lab.astype(np.uint8), cv2.COLOR_LAB2RGB)


def match_grain(inpainted, reference, mask, strength=0.4):
    blurred = cv2.GaussianBlur(reference.astype(np.float32), (5, 5), 0)
    grain = reference.astype(np.float32) - blurred
    mask_f = (mask[..., np.newaxis] / 255.0).astype(np.float32)
    return (inpainted.astype(np.float32) + grain * mask_f * strength).clip(0, 255).astype(np.uint8)


def match_motion_blur(inpainted, mask, flow, blur_scale=0.25):
    mask_bool = mask > 128
    if not mask_bool.any():
        return inpainted
    flow_mag = np.sqrt(flow[..., 0]**2 + flow[..., 1]**2)
    mean_flow = float(flow_mag[mask_bool].mean())
    ks = int(mean_flow * blur_scale)
    if ks < 1:
        return inpainted
    ks = min(ks | 1, 15)
    kernel = np.ones((5, 5), np.uint8)
    mask_dilated = cv2.dilate(mask, kernel, iterations=2)
    edge_mask = ((mask_dilated > 128) & ~mask_bool).astype(np.float32)[..., np.newaxis]
    blurred = cv2.GaussianBlur(inpainted, (ks, ks), 0)
    return (inpainted.astype(np.float32) * (1.0 - edge_mask) + blurred.astype(np.float32) * edge_mask).clip(0, 255).astype(np.uint8)


print("✅ Все функции готовы!")

## 4. Загрузка видео

Выбери один из способов: загрузить локальный файл или указать путь из Google Drive.

In [ ]:
# ── Вариант A: загрузить файл с компьютера ────────────────────────────
from google.colab import files

uploaded = files.upload()  # откроет диалог выбора файла

INPUT_VIDEO = list(uploaded.keys())[0]
print(f"\nЗагружено: {INPUT_VIDEO}  ({os.path.getsize(INPUT_VIDEO) / 1e6:.1f} MB)")

In [ ]:
# ── Вариант B: подключить Google Drive и указать путь ─────────────────
# (раскомментируй этот блок если хочешь использовать Drive)

# from google.colab import drive
# drive.mount('/content/drive')
# INPUT_VIDEO = "/content/drive/MyDrive/my_video.mp4"  # ← замени путь
# print(f"Видео: {INPUT_VIDEO}")

## 5. Параметры — настрой здесь!

Это главная ячейка с настройками. Измени параметры под свой случай.

In [ ]:
# ┌─────────────────────────────────────────────────────────────────────┐
# │                        НАСТРОЙКИ                                    │
# └─────────────────────────────────────────────────────────────────────┘

# Что найти на видео (используй простые слова: mug, watch, bottle, clock)
DETECT_PROMPT = "clock watch"

# Что сгенерировать вместо найденного объекта
PROMPT = "ceramic mug with space galaxy design, photorealistic, studio lighting"

# Что исключить из генерации
NEGATIVE_PROMPT = "distorted, unrealistic lighting, blurry, extra objects, ugly"

# Чувствительность детекции
# BOX_THRESHOLD:  0.35 — стандарт. Если объект не найден → снизь до 0.20.
#                 Если находит не то (стену, фон) → подними до 0.40-0.50.
# TEXT_THRESHOLD: обычно можно оставить 0.25.
BOX_THRESHOLD  = 0.35
TEXT_THRESHOLD = 0.25

# Сколько кадров проверить при поиске объекта по всему видео.
SCAN_FRAMES = 30

# ── Режим отладки ──────────────────────────────────────────────────────
# DEBUG_MAX_FRAMES: None = обрабатывать ВСЁ видео (правильный режим).
# Ставь число (напр. 60) ТОЛЬКО если хочешь быстро проверить результат
# на коротком фрагменте. В финальном запуске всегда оставляй None.
DEBUG_MAX_FRAMES = None

# Выходной файл
OUTPUT_VIDEO = "/content/output.mp4"

# Случайное зерно (влияет на стиль генерации)
SEED = 42

# Сила временного сглаживания (0.7 = плавнее, 1.0 = без сглаживания)
BLEND_ALPHA = 0.85

print("Параметры установлены:")
print(f"  Что найти:         '{DETECT_PROMPT}'")
print(f"  Что сгенерировать: '{PROMPT[:60]}...'")
print(f"  Порог детекции:    box={BOX_THRESHOLD}, text={TEXT_THRESHOLD}")
print(f"  Сканирование:      {SCAN_FRAMES} кадров по всему видео")
print(f"  Обработка:         {'ВСЁ видео' if DEBUG_MAX_FRAMES is None else f'только первые {DEBUG_MAX_FRAMES} кадров (DEBUG режим)'}")

## 6. Запуск пайплайна

Эта ячейка выполняет всю обработку. Время работы зависит от длины видео:
- 30 кадров ≈ 5-10 минут на T4 GPU
- 100 кадров ≈ 15-30 минут

In [ ]:
# ── Шаг 1: Поиск объекта по всему видео (без загрузки всех кадров) ────
# Поиск всегда идёт по ВСЕМУ видео, независимо от MAX_FRAMES.
print("=" * 60)
print("Шаг 1: Поиск объекта по всему видео (Grounding DINO)")
print("=" * 60)
print(f"  Ищу: '{DETECT_PROMPT}'  |  box_threshold={BOX_THRESHOLD}  text_threshold={TEXT_THRESHOLD}")

gdino = load_grounding_dino(GDINO_CHECKPOINT)
all_detections = scan_video_for_object(
    INPUT_VIDEO, gdino, DETECT_PROMPT, BOX_THRESHOLD, TEXT_THRESHOLD, SCAN_FRAMES
)

if not all_detections:
    raise RuntimeError(
        f"Объект '{DETECT_PROMPT}' не найден ни на одном из {SCAN_FRAMES} проверенных кадров.\n"
        f"Текущий BOX_THRESHOLD={BOX_THRESHOLD}. Попробуй:\n"
        f"  • Простые слова: 'mug', 'cup', 'watch', 'bottle', 'clock'\n"
        f"  • Снизь BOX_THRESHOLD до 0.20 в ячейке Параметры\n"
        f"  • Увеличь SCAN_FRAMES до 60"
    )

# Выбираем детекцию с максимальной уверенностью
all_detections.sort(reverse=True)  # по confidence (первый элемент кортежа)
best_conf, seed_frame_idx, found_bbox, found_phrase, seed_frame_rgb = all_detections[0]
print(f"\n  ★ Лучший результат: '{found_phrase}' на кадре {seed_frame_idx}  confidence={best_conf:.3f}")
if best_conf < 0.40:
    print(f"  ⚠ Уверенность невысокая ({best_conf:.3f}). Если маска выделила не тот объект,")
    print(f"    подними BOX_THRESHOLD до 0.40-0.50 и перезапусти.")

# ── Шаг 2: Извлечение ВСЕХ кадров видео ───────────────────────────────
# Загружаем всё видео с самого начала — объект может появляться и исчезать
# в любой момент, SAM2 Video Predictor отследит его по всем кадрам.
print("\n" + "=" * 60)
print("Шаг 2: Извлечение кадров видео")
print("=" * 60)

frames, fps = extract_frames(INPUT_VIDEO, max_frames=DEBUG_MAX_FRAMES, start_frame=0)

# seed_frame_idx совпадает с глобальным номером кадра (start_frame=0)
local_seed_idx = min(seed_frame_idx, len(frames) - 1)
if DEBUG_MAX_FRAMES and seed_frame_idx >= DEBUG_MAX_FRAMES:
    print(f"  ⚠ DEBUG_MAX_FRAMES={DEBUG_MAX_FRAMES}: seed-кадр {seed_frame_idx} вне диапазона.")
    print(f"    Используем кадр {local_seed_idx} как seed. Для полного видео поставь DEBUG_MAX_FRAMES=None.")
print(f"  Seed-кадр: {local_seed_idx} (глобальный {seed_frame_idx})")

# ── Шаг 3: Начальная маска (SAM2 Image) ───────────────────────────────
print("\n" + "=" * 60)
print("Шаг 3: Создание начальной маски (SAM2)")
print("=" * 60)

sam_image = build_sam2_image_predictor(SAM2_CHECKPOINT, SAM2_CONFIG)
initial_mask = get_mask_from_bbox(sam_image, frames[local_seed_idx], found_bbox)
del sam_image
torch.cuda.empty_cache()
print(f"  ✓ Маска создана. Пикселей в маске: {initial_mask.sum() // 255}")

# Показываем превью: bbox + маска
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(frames[local_seed_idx])
x1, y1, x2, y2 = found_bbox
rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='lime', linewidth=3)
axes[0].add_patch(rect)
axes[0].set_title(
    f"Кадр {seed_frame_idx} (глобальный) | '{found_phrase}' | conf={best_conf:.3f}\n"
    f"box_threshold={BOX_THRESHOLD}",
    fontsize=12,
)
axes[0].axis('off')

overlay = frames[local_seed_idx].copy()
overlay[initial_mask > 128] = [255, 80, 80]
blended_preview = (0.55 * frames[local_seed_idx] + 0.45 * overlay).astype(np.uint8)
axes[1].imshow(blended_preview)
axes[1].set_title("Маска SAM2 (красная зона = будет заменена)", fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.show()
print()
print("  ↑ ПРОВЕРЬ: красная зона должна покрывать нужный объект.")
print("  Если выделен не тот объект — подними BOX_THRESHOLD (сейчас", BOX_THRESHOLD, ").")
print("  Если объект не найден — снизь BOX_THRESHOLD или смени DETECT_PROMPT.")
print("  Измени параметры в ячейке 5 и перезапусти.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 🔍 ДИАГНОСТИКА: просмотр всех детекций по кадрам (опционально)
# Запусти эту ячейку отдельно если хочешь подобрать правильный порог.
# Она не влияет на основной пайплайн.
# ══════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Сканируем больше кадров с низким порогом чтобы увидеть все варианты
DIAG_THRESHOLD = 0.10  # специально низкий для диагностики

print(f"Диагностика: сканирую {min(20, len(frames))} кадров с порогом {DIAG_THRESHOLD}")
print(f"Ищу: '{DETECT_PROMPT}'")
print()

diag_indices = list(range(0, len(frames), max(1, len(frames) // 20)))[:20]
diag_results = []

for fi in diag_indices:
    result = detect_bbox(gdino, frames[fi], DETECT_PROMPT, DIAG_THRESHOLD, 0.10)
    if result is not None:
        if len(result) == 3:
            bbox, phrase, conf = result
        else:
            bbox, phrase = result
            conf = 0.5
        diag_results.append((fi, bbox, phrase, conf))

if not diag_results:
    print("Объект не найден даже с порогом 0.10 — попробуй другое слово в DETECT_PROMPT")
else:
    print(f"Найдено детекций: {len(diag_results)}")
    print()

    cols = min(4, len(diag_results))
    rows = (len(diag_results) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    if rows * cols == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]

    for idx, (fi, bbox, phrase, conf) in enumerate(diag_results):
        ax = axes[idx // cols][idx % cols]
        ax.imshow(frames[fi])
        x1, y1, x2, y2 = bbox
        color = 'lime' if conf >= BOX_THRESHOLD else 'orange'
        rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.set_title(f"Кадр {fi}\n'{phrase}'\nconf={conf:.3f}", fontsize=9,
                     color='green' if conf >= BOX_THRESHOLD else 'darkorange')
        ax.axis('off')

    # Скрываем пустые ячейки
    for idx in range(len(diag_results), rows * cols):
        axes[idx // cols][idx % cols].axis('off')

    plt.suptitle(
        f"Зелёная рамка = conf ≥ {BOX_THRESHOLD} (текущий порог) | "
        f"Оранжевая = ниже порога\n"
        f"Подбери BOX_THRESHOLD так, чтобы зелёными были только нужные объекты",
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Шаг 4: Отслеживание маски по всем кадрам (SAM2 Video) ─────────────
print("=" * 60)
print("Шаг 4: Отслеживание объекта по всем кадрам (SAM2 Video)")
print("=" * 60)

sam_video = build_sam2_video_predictor(SAM2_CHECKPOINT, SAM2_CONFIG)
tracked_masks = track_masks_video(sam_video, frames, initial_mask, local_seed_idx)
del sam_video
torch.cuda.empty_cache()

tracked_masks = fill_mask_gaps(tracked_masks, max_gap=2)
visible_count = sum(m.any() for m in tracked_masks)
print(f"  ✓ Отслеживание завершено. Объект виден в {visible_count}/{len(frames)} кадрах")

# ── Шаг 5: Загрузка Diffusion pipeline ────────────────────────────────
print("\n" + "=" * 60)
print("Шаг 5: Загрузка SDXL + ControlNet")
print("=" * 60)
print("  (SDXL ~6 GB — первый раз займёт несколько минут)")

pipe = load_pipeline()
del gdino  # освобождаем память
torch.cuda.empty_cache()

print("\n✅ Все модели загружены! Начинаю обработку кадров...")

In [ ]:
# ── Шаг 6: Обработка кадров ────────────────────────────────────────────
from IPython.display import clear_output, display
import time

print("=" * 60)
print("Шаг 6: Inpainting кадров")
print("=" * 60)
print(f"  Промпт: '{PROMPT}'")
print(f"  Всего кадров: {len(frames)}  |  С объектом: {visible_count}\n")

prev_result = None
processed = []
preview_frames = []  # для итогового превью
start_time = time.time()

for i, frame in enumerate(frames):
    mask = tracked_masks[i]

    if not mask.any():
        processed.append(frame.copy())
        prev_result = None
        print(f"  Кадр {i+1:3d}/{len(frames)} — объект не виден, пропускаю")
        continue

    # Оптический поток
    flow = compute_flow(frames[i-1], frame) if i > 0 else None

    # ControlNet Canny
    canny = make_canny(frame)

    # Inpainting
    inpainted = inpaint_frame(pipe, frame, mask, canny, PROMPT, NEGATIVE_PROMPT, seed=SEED)

    # Пост-обработка
    inpainted = apply_temporal_consistency(inpainted, prev_result, mask, flow, alpha=BLEND_ALPHA)
    inpainted = color_harmonize(frame, inpainted, mask)
    inpainted = match_grain(inpainted, frame, mask)
    if flow is not None:
        inpainted = match_motion_blur(inpainted, mask, flow)

    prev_result = inpainted.copy()
    processed.append(inpainted)

    if i < 5 or i % 10 == 0:  # сохраняем для превью
        preview_frames.append((i, frame.copy(), inpainted.copy()))

    elapsed = time.time() - start_time
    frames_done = i + 1
    fps_proc = frames_done / elapsed
    eta = (len(frames) - frames_done) / max(fps_proc, 0.001)
    print(f"  Кадр {frames_done:3d}/{len(frames)} ✓  [{elapsed:.0f}s прошло, ~{eta:.0f}s осталось]")

    torch.cuda.empty_cache()

total_time = time.time() - start_time
print(f"\n✅ Обработка завершена за {total_time/60:.1f} минут!")

In [ ]:
# ── Шаг 7: Сборка видео и превью ──────────────────────────────────────
print("Собираю выходное видео...")
imageio.mimsave(OUTPUT_VIDEO, processed, fps=fps)
size_mb = os.path.getsize(OUTPUT_VIDEO) / 1e6
print(f"✅ Видео сохранено: {OUTPUT_VIDEO}  ({size_mb:.1f} MB)")

# Показываем сравнение до/после
if preview_frames:
    fig, axes = plt.subplots(len(preview_frames), 2, figsize=(14, 5 * len(preview_frames)))
    if len(preview_frames) == 1:
        axes = [axes]
    for row, (fi, orig, result) in enumerate(preview_frames):
        axes[row][0].imshow(orig)
        axes[row][0].set_title(f"Кадр {fi+1} — ДО", fontsize=13)
        axes[row][0].axis('off')
        axes[row][1].imshow(result)
        axes[row][1].set_title(f"Кадр {fi+1} — ПОСЛЕ", fontsize=13)
        axes[row][1].axis('off')
    plt.suptitle("Сравнение: до и после", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Шаг 8: Скачать результат ──────────────────────────────────────────
from google.colab import files

print(f"Скачиваю {OUTPUT_VIDEO}...")
files.download(OUTPUT_VIDEO)
print("✅ Готово! Файл должен появиться в папке загрузок.")

---

## Советы и частые вопросы

### Объект не найден
- Попробуй более простое слово: `mug`, `cup`, `bottle`, `watch`, `phone`
- Снизь `BOX_THRESHOLD` до `0.10`–`0.15`
- Увеличь `SCAN_FRAMES` до 30

### Результат выглядит плохо
- Добавь больше деталей в `PROMPT` (материал, освещение, стиль)
- Поэкспериментируй с `SEED` (42, 123, 777)
- Измени `BLEND_ALPHA`: ближе к `0.7` = плавнее, ближе к `1.0` = чётче

### Мало памяти (OOM)
- Поставь `DEBUG_MAX_FRAMES = 60` в ячейке Параметры для теста на коротком фрагменте
- Runtime → Restart runtime → запусти заново
- После проверки верни `DEBUG_MAX_FRAMES = None` для обработки всего видео

### Сохранить в Google Drive
```python
import shutil
shutil.copy(OUTPUT_VIDEO, "/content/drive/MyDrive/output.mp4")
```